In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
from skimage.io import imread
from sklearn.model_selection import train_test_split
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Overview
- Simple object detector
- Input: RGB image
- Output: cars' locations

# Import libraries

In [ ]:
cars = glob.glob('./data/obj_detection/vehicles/*/*.png')
non_cars = glob.glob('./data/obj_detection/non-vehicles/*/*.png')
Y = np.concatenate([np.ones(len(cars)), np.zeros(len(non_cars))]).astype('float32')
X = np.array([imread(name) for name in cars + non_cars], dtype='float32')
X = torch.from_numpy(np.transpose(X, (0, 3, 1, 2))).div(127.5).sub(1).to(device)
Y = torch.from_numpy(Y).to(device)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.30, random_state=42)

# (1) Data

#### Car images: https://drive.google.com/file/d/1rqitUFrFhB84tFVtg1Ad1TVz6coCWpNZ/view?usp=sharing

In [ ]:
class VehicleDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(nn.Conv2d(3, 10, 3, padding=1), nn.ReLU(), nn.Conv2d(10, 10, 3, padding=1), nn.ReLU(), nn.MaxPool2d(8), nn.Dropout(0.25), nn.Conv2d(10, 128, 8), nn.ReLU(), nn.Dropout(0.5), nn.Conv2d(128, 1, 1))
        self.pool = nn.AdaptiveAvgPool2d(1)
    def forward(self, x): return self.pool(self.features(x)).flatten(1).squeeze(1)
model = VehicleDetector().to(device)
print(model)

In [ ]:
criterion = nn.MSELoss(); optimizer = torch.optim.Adam(model.parameters())
loader = DataLoader(TensorDataset(X_train, Y_train), batch_size=128, shuffle=True)
for epoch in range(5):
    model.train()
    for xb, yb in loader:
        optimizer.zero_grad(); loss = criterion(model(xb), yb); loss.backward(); optimizer.step()
    print(f'Epoch {epoch + 1}/5, loss={loss.item():.4f}')
torch.save(model.state_dict(), 'localize.pt')

# (2) Declare model

In [ ]:
model.eval()
with torch.no_grad():
    predictions = model(X_test)
print('Test MSE:', criterion(predictions, Y_test).item())

# (3) Train defined model

In [ ]:
def locate(data):
    model.eval()
    with torch.no_grad():
        tensor = torch.from_numpy(np.transpose(data[None].astype('float32'), (0, 3, 1, 2))).div(127.5).sub(1).to(device)
        return model(tensor).cpu().numpy()

# Example: heatmap = locate(imread('test.jpg'))

# (4) Evaluate trained model